In [ ]:
drop table vendor_pricing_data

In [ ]:
CREATE TABLE vendor_pricing_data (
    vendor_id VARCHAR,
    security_ticker VARCHAR,
    price DECIMAL(18, 6),
    as_of_date DATE,
    as_at_ts TIMESTAMP,
    PRIMARY KEY (vendor_id, security_ticker, as_of_date, as_at_ts)
);

In [ ]:
insert into
    vendor_pricing_data
values(
        'BB',
        'AAPL',
        274,
        '2025-12-15',
        '2025-12-15 18:00:00'
    ),
    (
        'BB',
        'AAPL',
        274,
        '2025-12-16',
        '2025-12-16 18:00:00'
    ),
    (
        'BB',
        'AAPL',
        271,
        '2025-12-17',
        '2025-12-18 18:00:00'
    ),
    (
        'BB',
        'AAPL',
        272,
        '2025-12-18',
        '2025-12-18 20:30:00'
    );

In [ ]:
insert into
    vendor_pricing_data
values(
        'BB',
        'AAPL',
        273,
        '2025-12-19',
        '2025-12-19 18:00:00'
    )

In [ ]:
SELECT vendor_id ,
    security_ticker,
    price,
    as_of_date,
    as_at_ts
FROM vendor_pricing_data
WHERE as_of_date = '2025-12-17'

In [ ]:
select * from vendor_pricing_data;

In [ ]:
DROP TABLE holding_snapshot;

In [ ]:

CREATE TABLE holding_snapshot(
investor_id INTEGER,
portfolio_name VARCHAR,
snapshot_ts TIMESTAMP,
security_ticker VARCHAR,
share_quantity DECIMAL(18,6),
PRIMARY KEY (investor_id, portfolio_name, snapshot_ts, security_ticker)
);

In [ ]:
INSERT INTO holding_snapshot VALUES (1001, 'Growth', '2025-12-16', 'AAPL', 200), (1001, 'Growth', '2025-12-17', 'AAPL', 220);

In [ ]:
INSERT INTO holding_snapshot VALUES (1001, 'Growth', '2025-12-18', 'AAPL', 220), (1001, 'Growth', '2025-12-19', 'AAPL', 190);

In [ ]:
truncate table holding_snapshot;

In [ ]:
INSERT INTO holding_snapshot VALUES (1001, 'Growth', '2025-12-16', 'AAPL', 200), (1001, 'Growth', '2025-12-17', 'AAPL', 220), (1001, 'Growth', '2025-12-18', 'AAPL', 210), (1001, 'Growth', '2025-12-19', 'AAPL', 190);

In [ ]:
select * from holding_snapshot

In [ ]:
WITH cte_price_fromto AS (
    SELECT 
        vendor_id,
        security_ticker,
        as_of_date,
        price,
        as_at_ts AS valid_from,
        LEAD(as_at_ts) OVER (PARTITION BY vendor_id, security_ticker ORDER BY as_at_ts ASC) AS valid_to
    FROM vendor_pricing_data
)
SELECT 
    hs.investor_id,
    hs.snapshot_ts, 
    hs.portfolio_name, 
    hs.security_ticker,
    hs.share_quantity,
    vp.price,
    vp.price * hs.share_quantity as ticker_value
FROM holding_snapshot hs
LEFT JOIN cte_price_fromto vp
    ON hs.security_ticker = vp.security_ticker
    AND hs.snapshot_ts >= vp.valid_from 
    AND (hs.snapshot_ts < vp.valid_to OR vp.valid_to IS NULL)
WHERE vp.vendor_id = 'BB'
AND hs.investor_id = 1001
AND hs.snapshot_ts = '2025-12-19 00:00:00' 


In [ ]:
SELECT 
    hs.investor_id, 
    hs.snapshot_ts, 
    hs.portfolio_name, 
    hs.security_ticker,
    vp.price,
    vp.price * hs.share_quantity  as ticker_value
FROM holding_snapshot hs
ASOF JOIN vendor_pricing_data vp
    MATCH_CONDITION (hs.snapshot_ts >= vp.as_at_ts )
    ON hs.security_ticker = vp.security_ticker
WHERE vp.vendor_id = 'BB'
AND hs.investor_id = 1001
AND hs.snapshot_ts = '2025-12-19 00:00:00' 